# BuzzSpot RF-DETR Large train+val final-fit, 32 epochs

First serious full-frame RF-DETR "Model A" run.

What this notebook does:
- uses current test-phase annotations from the tar
- uses train/valid images from the old zip
- trains RF-DETR Large on **official train + official valid annotated keyframes**
- uses the same rare-class full-frame oversampling recipe proven in the earlier experiments
- keeps a non-oversampled valid split only as a leaked sanity check
- trains for 32 fixed epochs with EMA enabled
- saves every epoch
- selects the best total checkpoint, preferring EMA when it performs better
- creates regular full-frame test submission first
- leaves SAHI inference as an optional later cell

The only intentional training changes from the successful 8-epoch RF-DETR trial are:
1. RF-DETR Small → RF-DETR Large
2. 8 epochs → 32 epochs

Important: validation AP in this notebook is leaked because valid images are also included in training. The leaderboard is the real signal.

- Uses a pinned RF-DETR/Pillow environment with a one-time runtime restart and early import smoke test.


## 1. Install
- Uses a pinned RF-DETR/Pillow environment with a one-time runtime restart and early import smoke test.


In [4]:
# Reproducible RF-DETR environment setup for Google Colab.
# This cell intentionally restarts the runtime ONCE after installation so Pillow
# and all compiled/imported modules are loaded from one consistent installation.

import os
import sys
import subprocess
from pathlib import Path

SETUP_MARKER = Path("/content/.buzzspot_rfdetr_env_ready_v3")

if not SETUP_MARKER.exists():
    print("Installing pinned RF-DETR environment...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "rfdetr[train,loggers]==1.8.3",
        "pycocotools",
        "tqdm",
        "supervision>=0.29.0",
    ])

    # Repair/prevent mixed Pillow files in long-lived Colab runtimes.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "--force-reinstall", "Pillow==12.3.0",
    ])

    SETUP_MARKER.write_text("ready")
    print("Environment installed. Restarting the Colab runtime once...")
    print("After reconnecting, run the notebook again from the top.")
    os.kill(os.getpid(), 9)
else:
    print("Pinned environment already installed; continuing without another restart.")


Pinned environment already installed; continuing without another restart.


## 1.1 Environment import smoke test

This runs before any dataset extraction so dependency problems fail immediately.


In [5]:
import PIL
import rfdetr
import supervision as sv
from PIL import Image, ImageText
from rfdetr import RFDETRLarge

print("Pillow:", PIL.__version__)
print("RF-DETR import: OK")
print("supervision:", sv.__version__)
print("RFDETRLarge class:", RFDETRLarge)


Pillow: 12.3.0
RF-DETR import: OK
supervision: 0.29.1
RFDETRLarge class: <class 'rfdetr.variants.RFDETRLarge'>


## 2. Mount Drive and locate dataset files

In [6]:
from google.colab import drive
drive.mount("/content/drive")

import glob
from pathlib import Path

zip_hits = glob.glob("/content/drive/MyDrive/**/BuzzSet_challenge.zip", recursive=True)
assert zip_hits, "BuzzSet_challenge.zip not found under MyDrive"
ZIP_PATH = zip_hits[0]

tar_hits = glob.glob("/content/drive/MyDrive/**/19557529600-BuzzSet_challenge_testphase.tar", recursive=True)
if not tar_hits:
    tar_hits = glob.glob("/content/drive/MyDrive/**/*BuzzSet_challenge_testphase*.tar*", recursive=True)
assert tar_hits, "test-phase tar file not found under MyDrive"
TAR_PATH = tar_hits[0]

print("old train/valid zip:", ZIP_PATH)
print("new test-phase tar:", TAR_PATH)


Mounted at /content/drive
old train/valid zip: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/BuzzSet_challenge.zip
new test-phase tar: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/19557529600-BuzzSet_challenge_testphase.tar


## 3. Config

In [7]:
from pathlib import Path
import shutil

# Main switch
ENABLE_TRAINING = False  # First serious RF-DETR Large Model A training run.

# Inference/eval switches
ENABLE_REGULAR_VALIDATION = True       # leaked sanity only
ENABLE_REGULAR_INFERENCE = True
ENABLE_SAHI_INFERENCE = False          # optional; regular full-frame is the first overnight submission

MODEL_SIZE = "large"
MODEL_TAG = "rfdetr_large_trainval_os_1120_32ep"
RUN_NAME = "buzz_rfdetr_large_trainval_os_1120_32ep"

# RF-DETR setup
RESOLUTION = 1120       # safer overnight high-res choice. If OOM: 896 or batch_size=1/grad_accum=16.
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8    # effective batch = 16
EPOCHS = 32
LR = 1e-4
LR_ENCODER = 1e-4       # unchanged from the successful RF-DETR trial
WARMUP_EPOCHS = 1.0
USE_EMA = True
EARLY_STOPPING = False  # fixed 32-epoch run; do not stop on leaked valid
GRADIENT_CHECKPOINTING = True
CHECKPOINT_INTERVAL = 1
EVAL_INTERVAL = 1
DEVICE = "cuda"
SEED = 0

# Rare-class oversampling. For multi-class images, use the max multiplier.
# 1 bee, 2 bumblebee, 3 hoverfly, 4 moth
CATEGORY_MULTIPLIERS = {
    1: 1,
    2: 4,
    3: 2,
    4: 5,
}

# Regular RF-DETR inference. Keep low for AP-style ranked detections.
REGULAR_CONF = 0.001

# Optional manual SAHI inference settings. Disabled by default.
SAHI_CONF = 0.05
SAHI_SLICE_SIZE = 700
SAHI_OVERLAP = 0.20
SAHI_MATCH_THRESHOLD = 0.50
SAHI_MAX_DET_PER_IMAGE = 300

PROJECT_DIR = Path("/content/drive/MyDrive/BuzzSpot")
LOCAL_DATA_DIR = Path("/content/buzz_rfdetr_trainval_os")
DRIVE_RUNS_DIR = PROJECT_DIR / "runs_rfdetr"
WEIGHTS_DIR = PROJECT_DIR / "weights"
SUBMISSIONS_DIR = PROJECT_DIR / "submissions"

for d in [DRIVE_RUNS_DIR, WEIGHTS_DIR, SUBMISSIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DRIVE_RUN_DIR = DRIVE_RUNS_DIR / RUN_NAME
DRIVE_SELECTED_WEIGHTS = WEIGHTS_DIR / f"{MODEL_TAG}_selected_for_inference.pth"
LOCAL_WEIGHTS = Path("/content/rfdetr_selected_for_inference.pth")

REG_TAG = f"regular_conf{int(REGULAR_CONF * 1000):03d}"
SAHI_TAG = f"sahi_conf{int(SAHI_CONF * 1000):03d}_slice{SAHI_SLICE_SIZE}_ov{int(SAHI_OVERLAP*100):02d}"

LOCAL_REG_PRED_PATH = Path(f"/content/predictions_{MODEL_TAG}_{REG_TAG}.json")
LOCAL_REG_ZIP_OUT = Path(f"/content/submission_{MODEL_TAG}_{REG_TAG}.zip")
DRIVE_REG_PRED_PATH = SUBMISSIONS_DIR / LOCAL_REG_PRED_PATH.name
DRIVE_REG_ZIP_OUT = SUBMISSIONS_DIR / LOCAL_REG_ZIP_OUT.name

LOCAL_SAHI_PRED_PATH = Path(f"/content/predictions_{MODEL_TAG}_{SAHI_TAG}.json")
LOCAL_SAHI_ZIP_OUT = Path(f"/content/submission_{MODEL_TAG}_{SAHI_TAG}.zip")
DRIVE_SAHI_PRED_PATH = SUBMISSIONS_DIR / LOCAL_SAHI_PRED_PATH.name
DRIVE_SAHI_ZIP_OUT = SUBMISSIONS_DIR / LOCAL_SAHI_ZIP_OUT.name

CLASS_NAMES = ["bee", "bumblebee", "hoverfly", "moth"]
CATEGORIES = [{"id": i + 1, "name": name, "supercategory": "pollinator"} for i, name in enumerate(CLASS_NAMES)]

print("ENABLE_TRAINING:", ENABLE_TRAINING)
print("MODEL_TAG:", MODEL_TAG)
print("RESOLUTION:", RESOLUTION)
print("BATCH_SIZE:", BATCH_SIZE)
print("GRAD_ACCUM_STEPS:", GRAD_ACCUM_STEPS)
print("effective batch:", BATCH_SIZE * GRAD_ACCUM_STEPS)
print("EPOCHS:", EPOCHS)
print("CATEGORY_MULTIPLIERS:", CATEGORY_MULTIPLIERS)
print("DRIVE_RUN_DIR:", DRIVE_RUN_DIR)
print("REGULAR ZIP:", DRIVE_REG_ZIP_OUT)


ENABLE_TRAINING: False
MODEL_TAG: rfdetr_large_trainval_os_1120_32ep
RESOLUTION: 1120
BATCH_SIZE: 2
GRAD_ACCUM_STEPS: 8
effective batch: 16
EPOCHS: 32
CATEGORY_MULTIPLIERS: {1: 1, 2: 4, 3: 2, 4: 5}
DRIVE_RUN_DIR: /content/drive/MyDrive/BuzzSpot/runs_rfdetr/buzz_rfdetr_large_trainval_os_1120_32ep
REGULAR ZIP: /content/drive/MyDrive/BuzzSpot/submissions/submission_rfdetr_large_trainval_os_1120_32ep_regular_conf001.zip


## 4. Build RF-DETR COCO dataset

In [8]:
import json, zipfile, tarfile, shutil, random
from pathlib import Path
from collections import defaultdict, Counter
from tqdm.auto import tqdm

random.seed(SEED)

if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)

for split in ["train", "valid", "test"]:
    (LOCAL_DATA_DIR / split).mkdir(parents=True, exist_ok=True)

zf = zipfile.ZipFile(ZIP_PATH)
tf = tarfile.open(TAR_PATH, "r:*")

zip_names = set(zf.namelist())
tar_members = {m.name: m for m in tf.getmembers() if m.isfile()}
tar_names = set(tar_members.keys())

def find_zip(rel):
    rel = rel.lstrip("/")
    for cand in (rel, f"BuzzSet_challenge/{rel}"):
        if cand in zip_names:
            return cand
    suffix = "/" + rel
    hits = [n for n in zip_names if n.endswith(suffix)]
    return hits[0] if hits else None

def find_tar(rel):
    rel = rel.lstrip("/")
    for cand in (rel, f"BuzzSet_challenge/{rel}", f"BuzzSet_challenge_testphase/{rel}"):
        if cand in tar_names:
            return cand
    suffix = "/" + rel
    hits = [n for n in tar_names if n.endswith(suffix)]
    return hits[0] if hits else None

def load_tar_json(fname):
    p = find_tar(f"annotations/{fname}") or find_tar(fname)
    assert p is not None, f"{fname} not found in tar"
    with tf.extractfile(tar_members[p]) as f:
        obj = json.load(f)
    print("loaded current annotation:", fname, "from", p)
    return obj

def flat_name(file_name):
    return file_name.replace("/", "__")

def clip_bbox_xywh(bbox, W, H):
    x, y, w, h = map(float, bbox)
    x = max(0.0, min(x, W - 1))
    y = max(0.0, min(y, H - 1))
    w = max(0.0, min(w, W - x))
    h = max(0.0, min(h, H - y))
    if w < 1 or h < 1:
        return None
    return [round(x, 2), round(y, 2), round(w, 2), round(h, 2)]

def anns_by_image(coco):
    by_img = defaultdict(list)
    for ann in coco.get("annotations", []):
        by_img[int(ann["image_id"])].append(ann)
    return by_img

def get_annotated_records(coco, zip_img_dir, source_split_name):
    by_img = anns_by_image(coco)
    annotated_ids = set(by_img.keys())
    records = []

    skipped_unannotated = 0
    class_counts = Counter()

    for im in coco["images"]:
        iid = int(im["id"])
        if iid not in annotated_ids:
            skipped_unannotated += 1
            continue

        W, H = int(im["width"]), int(im["height"])
        src = find_zip(f"{zip_img_dir}/{im['file_name']}") or find_zip(im["file_name"])
        assert src is not None, f"missing image in old zip: {zip_img_dir}/{im['file_name']}"

        anns = []
        for ann in by_img[iid]:
            bbox = clip_bbox_xywh(ann["bbox"], W, H)
            if bbox is None:
                continue
            cat = int(ann["category_id"])
            assert cat in {1, 2, 3, 4}, f"bad category_id: {cat}"
            anns.append({
                "category_id": cat,
                "bbox": bbox,
                "iscrowd": int(ann.get("iscrowd", 0)),
            })
            class_counts[cat] += 1

        if anns:
            records.append({
                "source_split": source_split_name,
                "orig_id": iid,
                "file_name": im["file_name"],
                "width": W,
                "height": H,
                "zip_member": src,
                "anns": anns,
            })

    print()
    print(f"{source_split_name} COCO image records:", len(coco["images"]))
    print(f"{source_split_name} annotated image ids used:", len(records))
    print(f"{source_split_name} skipped unannotated/context image records:", skipped_unannotated)
    print(f"{source_split_name} boxes:", sum(len(r["anns"]) for r in records))
    print(f"{source_split_name} class counts:", dict(class_counts))
    print(f"{source_split_name} named counts:", {CLASS_NAMES[k-1]: class_counts[k] for k in range(1, 5)})
    return records

def copy_zip_member(member, dst):
    with zf.open(member) as s, open(dst, "wb") as d:
        shutil.copyfileobj(s, d)

def build_train_oversampled(records):
    out_images = []
    out_annotations = []
    source_class_counts = Counter()
    effective_class_counts = Counter()
    multiplier_counts = Counter()
    next_image_id = 1
    next_ann_id = 1

    for rec in tqdm(records, desc="build oversampled train"):
        cats = {int(a["category_id"]) for a in rec["anns"]}
        mult = max(CATEGORY_MULTIPLIERS.get(c, 1) for c in cats)
        multiplier_counts[mult] += 1

        for a in rec["anns"]:
            source_class_counts[int(a["category_id"])] += 1

        for rep in range(mult):
            new_img_id = next_image_id
            next_image_id += 1

            stem = flat_name(Path(rec["file_name"]).with_suffix("").as_posix())
            suffix = Path(rec["file_name"]).suffix or ".jpg"
            out_name = f"{rec['source_split']}__orig{rec['orig_id']:06d}__rep{rep:02d}__{stem}{suffix}"
            out_path = LOCAL_DATA_DIR / "train" / out_name

            copy_zip_member(rec["zip_member"], out_path)

            out_images.append({
                "id": new_img_id,
                "file_name": out_name,
                "width": rec["width"],
                "height": rec["height"],
            })

            for a in rec["anns"]:
                cat = int(a["category_id"])
                bbox = list(a["bbox"])
                out_annotations.append({
                    "id": next_ann_id,
                    "image_id": new_img_id,
                    "category_id": cat,
                    "bbox": bbox,
                    "area": round(bbox[2] * bbox[3], 2),
                    "iscrowd": int(a.get("iscrowd", 0)),
                })
                next_ann_id += 1
                effective_class_counts[cat] += 1

    out_coco = {
        "info": {"description": "BuzzSpot RF-DETR train+valid annotated keyframes with rare-class oversampling"},
        "licenses": [],
        "categories": CATEGORIES,
        "images": out_images,
        "annotations": out_annotations,
    }
    ann_path = LOCAL_DATA_DIR / "train" / "_annotations.coco.json"
    ann_path.write_text(json.dumps(out_coco))

    print()
    print("TRAIN OVERSAMPLED")
    print("source unique images:", len(records))
    print("final train images:", len(out_images))
    print("final train boxes:", len(out_annotations))
    print("multiplier counts:", dict(multiplier_counts))
    print("source instance counts:", {CLASS_NAMES[k-1]: source_class_counts[k] for k in range(1, 5)})
    print("effective instance counts:", {CLASS_NAMES[k-1]: effective_class_counts[k] for k in range(1, 5)})
    print("train annotations:", ann_path)
    return out_coco

def build_valid_split(records):
    out_images = []
    out_annotations = []
    class_counts = Counter()
    next_ann_id = 1

    for rec in tqdm(records, desc="build valid sanity split"):
        out_name = f"valid__orig{rec['orig_id']:06d}__{flat_name(rec['file_name'])}"
        out_path = LOCAL_DATA_DIR / "valid" / out_name
        copy_zip_member(rec["zip_member"], out_path)

        out_images.append({
            "id": int(rec["orig_id"]),
            "file_name": out_name,
            "width": rec["width"],
            "height": rec["height"],
        })

        for a in rec["anns"]:
            cat = int(a["category_id"])
            bbox = list(a["bbox"])
            out_annotations.append({
                "id": next_ann_id,
                "image_id": int(rec["orig_id"]),
                "category_id": cat,
                "bbox": bbox,
                "area": round(bbox[2] * bbox[3], 2),
                "iscrowd": int(a.get("iscrowd", 0)),
            })
            next_ann_id += 1
            class_counts[cat] += 1

    out_coco = {
        "info": {"description": "BuzzSpot valid annotated keyframes, leaked sanity split only"},
        "licenses": [],
        "categories": CATEGORIES,
        "images": out_images,
        "annotations": out_annotations,
    }
    ann_path = LOCAL_DATA_DIR / "valid" / "_annotations.coco.json"
    ann_path.write_text(json.dumps(out_coco))

    print()
    print("VALID SANITY SPLIT, NOT OVERSAMPLED")
    print("valid images:", len(out_images))
    print("valid boxes:", len(out_annotations))
    print("valid named counts:", {CLASS_NAMES[k-1]: class_counts[k] for k in range(1, 5)})
    print("valid annotations:", ann_path)
    return out_coco

def build_test_split(test_coco):
    keyframe_images = [
        im for im in test_coco["images"]
        if im.get("is_keyframe") in [True, 1, "true", "True"]
    ]
    if not keyframe_images:
        raise RuntimeError("No test keyframes found. Check test_testphase.json is_keyframe flags.")

    out_images = []
    flat_to_id = {}
    image_id_to_path = {}

    for im in tqdm(keyframe_images, desc="extract test_testphase keyframes"):
        iid = int(im["id"])
        src = find_tar(f"test_testphase/{im['file_name']}") or find_tar(im["file_name"])
        assert src is not None, f"missing test image in tar: test_testphase/{im['file_name']}"

        out_name = flat_name(im["file_name"])
        dst = LOCAL_DATA_DIR / "test" / out_name
        with tf.extractfile(tar_members[src]) as s, open(dst, "wb") as d:
            shutil.copyfileobj(s, d)

        out_images.append({
            "id": iid,
            "file_name": out_name,
            "width": int(im.get("width", 1920)),
            "height": int(im.get("height", 1080)),
        })
        flat_to_id[out_name] = iid
        image_id_to_path[iid] = dst

    out_coco = {
        "info": {"description": "BuzzSpot test_testphase keyframes for RF-DETR submission"},
        "licenses": [],
        "categories": CATEGORIES,
        "images": out_images,
        "annotations": [],
    }
    ann_path = LOCAL_DATA_DIR / "test" / "_annotations.coco.json"
    ann_path.write_text(json.dumps(out_coco))

    print()
    print("TEST")
    print("test_testphase keyframes:", len(out_images))
    print("test annotations:", ann_path)
    return out_coco, flat_to_id, image_id_to_path

train_json = load_tar_json("train.json")
valid_json = load_tar_json("valid.json")
test_json = load_tar_json("test_testphase.json")

valid_records = get_annotated_records(valid_json, "valid", "valid")

if ENABLE_TRAINING:
    train_records = get_annotated_records(train_json, "train", "train")
    train_source_records = train_records + valid_records

    print()
    print("FINAL-FIT RF-DETR TRAIN SOURCE")
    print("train annotated images:", len(train_records))
    print("valid annotated images added:", len(valid_records))
    print("train+valid annotated images:", len(train_source_records))

    train_coco = build_train_oversampled(train_source_records)
else:
    print()
    print("ENABLE_TRAINING=False -> skipping expensive train+valid oversampled dataset rebuild.")
    train_coco = None

valid_coco = build_valid_split(valid_records)
test_coco, flat_to_id, test_image_id_to_path = build_test_split(test_json)

keyframe_ids = set(flat_to_id.values())

print()
print("RF-DETR dataset dir:", LOCAL_DATA_DIR)
if ENABLE_TRAINING:
    print("train annotations:", LOCAL_DATA_DIR / "train" / "_annotations.coco.json")
print("valid annotations:", LOCAL_DATA_DIR / "valid" / "_annotations.coco.json")
print("test annotations:", LOCAL_DATA_DIR / "test" / "_annotations.coco.json")


loaded current annotation: train.json from BuzzSpot_testphase/annotations/train.json
loaded current annotation: valid.json from BuzzSpot_testphase/annotations/valid.json
loaded current annotation: test_testphase.json from BuzzSpot_testphase/annotations/test_testphase.json

valid COCO image records: 5592
valid annotated image ids used: 932
valid skipped unannotated/context image records: 4660
valid boxes: 1116
valid class counts: {1: 934, 2: 30, 3: 95, 4: 57}
valid named counts: {'bee': 934, 'bumblebee': 30, 'hoverfly': 95, 'moth': 57}

ENABLE_TRAINING=False -> skipping expensive train+valid oversampled dataset rebuild.


build valid sanity split:   0%|          | 0/932 [00:00<?, ?it/s]


VALID SANITY SPLIT, NOT OVERSAMPLED
valid images: 932
valid boxes: 1116
valid named counts: {'bee': 934, 'bumblebee': 30, 'hoverfly': 95, 'moth': 57}
valid annotations: /content/buzz_rfdetr_trainval_os/valid/_annotations.coco.json


extract test_testphase keyframes:   0%|          | 0/4763 [00:00<?, ?it/s]


TEST
test_testphase keyframes: 4763
test annotations: /content/buzz_rfdetr_trainval_os/test/_annotations.coco.json

RF-DETR dataset dir: /content/buzz_rfdetr_trainval_os
valid annotations: /content/buzz_rfdetr_trainval_os/valid/_annotations.coco.json
test annotations: /content/buzz_rfdetr_trainval_os/test/_annotations.coco.json


## 5. Train RF-DETR Large or restore checkpoint


In [9]:
import shutil
from pathlib import Path

def get_rfdetr_class():
    if MODEL_SIZE.lower() == "small":
        from rfdetr import RFDETRSmall
        return RFDETRSmall
    elif MODEL_SIZE.lower() == "medium":
        from rfdetr import RFDETRMedium
        return RFDETRMedium
    elif MODEL_SIZE.lower() == "base":
        from rfdetr import RFDETRBase
        return RFDETRBase
    elif MODEL_SIZE.lower() == "large":
        from rfdetr import RFDETRLarge
        return RFDETRLarge
    else:
        raise ValueError(f"Unsupported MODEL_SIZE: {MODEL_SIZE}")

RFDETRClass = get_rfdetr_class()

def checkpoint_candidates():
    candidates = []
    # Prefer the best total checkpoint (EMA when it outperforms regular weights).
    # If it cannot be loaded for inference, fall back to the other saved checkpoints.
    for name in [
        "checkpoint_best_total.pth",
        "checkpoint_best_ema.pth",
        "checkpoint_best_regular.pth",
        f"checkpoint_{EPOCHS}.pth",
        "checkpoint.pth",
    ]:
        p = DRIVE_RUN_DIR / name
        if p.exists():
            candidates.append(p)

    if DRIVE_SELECTED_WEIGHTS.exists():
        candidates.append(DRIVE_SELECTED_WEIGHTS)

    for p in sorted(DRIVE_RUN_DIR.glob("checkpoint_*.pth")) if DRIVE_RUN_DIR.exists() else []:
        if p not in candidates:
            candidates.append(p)

    return candidates

def try_load_checkpoint(path):
    m = RFDETRClass.from_checkpoint(str(path))
    print("Smoke-test model resolution:", m.model_config.resolution)
    assert m.model_config.resolution == RESOLUTION
    assert m.model.resolution == RESOLUTION
    # Tiny smoke test on one test image path catches incompatible checkpoint formats.
    sample_id = next(iter(test_image_id_to_path))
    _ = m.predict(str(test_image_id_to_path[sample_id]), threshold=0.5)
    return m

def select_and_copy_checkpoint():
    cands = checkpoint_candidates()
    assert cands, f"No RF-DETR checkpoints found in {DRIVE_RUN_DIR}"

    errors = []
    for p in cands:
        try:
            print("Trying checkpoint:", p)
            _ = try_load_checkpoint(p)
            shutil.copy(p, DRIVE_SELECTED_WEIGHTS)
            shutil.copy(p, LOCAL_WEIGHTS)
            print("Selected checkpoint:", p)
            print("Backed up selected checkpoint to:", DRIVE_SELECTED_WEIGHTS)
            print("Copied local checkpoint:", LOCAL_WEIGHTS)
            return LOCAL_WEIGHTS
        except Exception as e:
            errors.append((str(p), repr(e)))
            print("Could not load for inference:", p)
            print("Error:", repr(e))

    raise RuntimeError("No checkpoint candidate could be loaded. Errors:\n" + "\n".join(f"{p}: {e}" for p, e in errors))

def restore_checkpoint_to_local():
    if DRIVE_SELECTED_WEIGHTS.exists():
        shutil.copy(DRIVE_SELECTED_WEIGHTS, LOCAL_WEIGHTS)
        print("Restored selected checkpoint from:", DRIVE_SELECTED_WEIGHTS)
        return LOCAL_WEIGHTS
    return select_and_copy_checkpoint()

if ENABLE_TRAINING:
    print(f"ENABLE_TRAINING=True -> training RF-DETR {MODEL_SIZE.title()} final-fit.")
    DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)

    model = RFDETRClass()
    train_kwargs = dict(
        dataset_dir=str(LOCAL_DATA_DIR),
        output_dir=str(DRIVE_RUN_DIR),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        lr=LR,
        lr_encoder=LR_ENCODER,
        resolution=RESOLUTION,
        use_ema=USE_EMA,
        early_stopping=EARLY_STOPPING,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        checkpoint_interval=CHECKPOINT_INTERVAL,
        eval_interval=EVAL_INTERVAL,
        eval_max_dets=500,
        log_per_class_metrics=True,
        warmup_epochs=WARMUP_EPOCHS,
        device=DEVICE,
        seed=SEED,
    )
    print("training args:", train_kwargs)
    model.train(**train_kwargs)

    select_and_copy_checkpoint()
else:
    print("ENABLE_TRAINING=False -> skipping training.")
    restore_checkpoint_to_local()


ENABLE_TRAINING=False -> skipping training.
Restored selected checkpoint from: /content/drive/MyDrive/BuzzSpot/weights/rfdetr_large_trainval_os_1120_32ep_selected_for_inference.pth


## 6. Load selected RF-DETR checkpoint

In [10]:
import torch, gc

restore_checkpoint_to_local()
RFDETRClass = get_rfdetr_class()
model = RFDETRClass.from_checkpoint(str(LOCAL_WEIGHTS))

print("Loaded RF-DETR checkpoint:", LOCAL_WEIGHTS)
print("Model resolution:", model.model_config.resolution)

assert model.model_config.resolution == RESOLUTION
assert model.model.resolution == RESOLUTION

gc.collect()
torch.cuda.empty_cache()


[2026-07-06 04:53:35] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-06 04:53:35] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


Restored selected checkpoint from: /content/drive/MyDrive/BuzzSpot/weights/rfdetr_large_trainval_os_1120_32ep_selected_for_inference.pth


[2026-07-06 04:53:35] [WARNING] rf-detr - load_pretrain_weights: args.num_queries absent; inferred ckpt_num_queries=300 from tensor rows 3900 ÷ ckpt_group_detr=13.


Loaded RF-DETR checkpoint: /content/rfdetr_selected_for_inference.pth
Model resolution: 1120


## 7. Prediction helpers and robust custom-class mapping

RF-DETR fine-tuned checkpoints expose custom class names in
`detections.data["class_name"]`. This notebook maps those names directly to
BuzzSpot category IDs and ignores any placeholder/background class instead of
guessing whether numeric IDs are zero- or one-based.


In [11]:
import json, zipfile, shutil, math, gc, os, tempfile
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
import numpy as np
import torch
from collections import Counter

W_FULL, H_FULL = 1920, 1080

CLASS_NAME_TO_CATEGORY_ID = {
    "bee": 1,
    "bumblebee": 2,
    "hoverfly": 3,
    "moth": 4,
}

def normalize_class_name(value):
    if value is None:
        return None
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="replace")
    return str(value).strip().lower().replace("_", "").replace("-", "").replace(" ", "")

NORMALIZED_CLASS_NAME_TO_CATEGORY_ID = {
    normalize_class_name(name): category_id
    for name, category_id in CLASS_NAME_TO_CATEGORY_ID.items()
}

def detection_class_names(detections):
    data = getattr(detections, "data", None)
    if not isinstance(data, dict):
        return None

    names = data.get("class_name")
    if names is None:
        return None

    names = np.asarray(names, dtype=object)
    return names

def inspect_detection_schema(model, image_id_to_path, threshold=0.001, max_images=50):
    seen = Counter()

    for idx, (_, img_path) in enumerate(image_id_to_path.items()):
        if idx >= max_images:
            break

        detections = model.predict(str(img_path), threshold=threshold)
        class_ids = getattr(detections, "class_id", None)
        class_names = detection_class_names(detections)

        if class_ids is None:
            continue

        class_ids = np.asarray(class_ids)
        if class_names is None or len(class_names) != len(class_ids):
            raise RuntimeError(
                "Fine-tuned RF-DETR predictions did not include a matching "
                'detections.data["class_name"] array. Refusing to guess numeric class IDs.'
            )

        for class_id, class_name in zip(class_ids, class_names):
            seen[(int(class_id), str(class_name))] += 1

    print("RF-DETR raw class-id / class-name pairs seen in probe:")
    for pair, count in seen.most_common():
        print(f"  {pair}: {count}")

    resolved_names = {
        normalize_class_name(class_name)
        for (_, class_name) in seen
        if normalize_class_name(class_name) in NORMALIZED_CLASS_NAME_TO_CATEGORY_ID
    }
    print("Resolved BuzzSpot class names:", sorted(resolved_names))

    if not resolved_names:
        raise RuntimeError(
            "The checkpoint probe did not expose any recognized BuzzSpot class names."
        )

inspect_detection_schema(
    model,
    test_image_id_to_path,
    threshold=REGULAR_CONF,
    max_images=50,
)

def category_id_from_detection(class_name):
    normalized = normalize_class_name(class_name)
    return NORMALIZED_CLASS_NAME_TO_CATEGORY_ID.get(normalized)

def clip_xyxy(x1, y1, x2, y2, W, H):
    x1 = max(0.0, min(float(x1), W - 1))
    y1 = max(0.0, min(float(y1), H - 1))
    x2 = max(0.0, min(float(x2), W))
    y2 = max(0.0, min(float(y2), H))

    if x2 - x1 < 1 or y2 - y1 < 1:
        return None

    return x1, y1, x2, y2

def detections_to_records(
    detections,
    image_id,
    x_offset=0,
    y_offset=0,
    W=W_FULL,
    H=H_FULL,
    max_det=None,
):
    records = []

    if detections is None:
        return records

    xyxy = getattr(detections, "xyxy", None)
    conf = getattr(detections, "confidence", None)
    class_ids = getattr(detections, "class_id", None)
    class_names = detection_class_names(detections)

    if xyxy is None or conf is None or class_ids is None:
        return records

    xyxy = np.asarray(xyxy)
    conf = np.asarray(conf)
    class_ids = np.asarray(class_ids)

    if class_names is None or len(class_names) != len(class_ids):
        raise RuntimeError(
            'Missing or mismatched detections.data["class_name"]. '
            "Numeric RF-DETR class IDs are intentionally not guessed."
        )

    order = np.argsort(-conf)
    if max_det is not None:
        order = order[:max_det]

    skipped_unknown = 0

    for j in order:
        category_id = category_id_from_detection(class_names[j])

        # RF-DETR may emit a placeholder/background class at a very low threshold.
        # Ignore anything whose checkpoint class name is not one of the four BuzzSpot classes.
        if category_id is None:
            skipped_unknown += 1
            continue

        x1, y1, x2, y2 = xyxy[j].tolist()
        clipped = clip_xyxy(
            x1 + x_offset,
            y1 + y_offset,
            x2 + x_offset,
            y2 + y_offset,
            W,
            H,
        )

        if clipped is None:
            continue

        x1, y1, x2, y2 = clipped
        records.append({
            "image_id": int(image_id),
            "category_id": int(category_id),
            "bbox": [
                round(x1, 2),
                round(y1, 2),
                round(x2 - x1, 2),
                round(y2 - y1, 2),
            ],
            "score": round(float(conf[j]), 5),
        })

    return records

def predict_one_image_regular(model, img_path, image_id, threshold):
    detections = model.predict(str(img_path), threshold=threshold)
    return detections_to_records(
        detections,
        image_id=image_id,
        max_det=500,
    )

def write_submission(
    preds,
    pred_path,
    zip_path,
    drive_pred_path=None,
    drive_zip_path=None,
):
    with open(pred_path, "w") as f:
        json.dump(preds, f)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(pred_path, arcname="predictions.json")

    if drive_pred_path is not None:
        shutil.copy(pred_path, drive_pred_path)

    if drive_zip_path is not None:
        shutil.copy(zip_path, drive_zip_path)

def validate_submission_file(pred_path, zip_path):
    preds = json.loads(Path(pred_path).read_text())

    with zipfile.ZipFile(zip_path, "r") as archive:
        names = archive.namelist()

    assert names == ["predictions.json"], names

    bad_image_ids = [
        pred for pred in preds
        if int(pred["image_id"]) not in keyframe_ids
    ]
    bad_categories = [
        pred for pred in preds
        if int(pred["category_id"]) not in {1, 2, 3, 4}
    ]
    bad_scores = [
        pred for pred in preds
        if not (0.0 <= float(pred["score"]) <= 1.0)
    ]

    assert not bad_image_ids, f"Predictions outside test keyframes: {bad_image_ids[:5]}"
    assert not bad_categories, f"Invalid categories: {bad_categories[:5]}"
    assert not bad_scores, f"Invalid scores: {bad_scores[:5]}"

    degenerate = []
    out_of_bounds = []

    for pred in preds:
        x, y, w, h = map(float, pred["bbox"])

        if w <= 0 or h <= 0:
            degenerate.append(pred)
            continue

        if (
            x < 0
            or y < 0
            or x + w > W_FULL + 1e-6
            or y + h > H_FULL + 1e-6
        ):
            out_of_bounds.append(pred)

    assert not degenerate, f"Degenerate boxes: {degenerate[:5]}"
    assert not out_of_bounds, f"Out-of-bounds boxes: {out_of_bounds[:5]}"

    counts = Counter(int(pred["category_id"]) for pred in preds)

    print("submission validation passed")
    print("detections:", len(preds))
    print("predicted images:", len({int(pred["image_id"]) for pred in preds}), "/", len(keyframe_ids))
    print("class counts:", {
        CLASS_NAMES[category_id - 1]: counts[category_id]
        for category_id in range(1, 5)
    })
    print("zip contents:", names)


[2026-07-06 04:53:36] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.optimize_for_inference(dtype=torch.float16).


RF-DETR raw class-id / class-name pairs seen in probe:
  (0, 'bee'): 8793
  (1, 'bumblebee'): 2689
  (2, 'hoverfly'): 2116
  (3, 'moth'): 1402
Resolved BuzzSpot class names: ['bee', 'bumblebee', 'hoverfly', 'moth']


## 8. Optional leaked sanity validation on the valid split

This is post-training only. It does not retrain the model and it is not an honest generalization estimate because these images were included in training.


In [12]:
if ENABLE_REGULAR_VALIDATION:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    print("WARNING: This valid AP is leaked because valid images were included in training.")
    val_ann_path = LOCAL_DATA_DIR / "valid" / "_annotations.coco.json"
    val_coco = json.load(open(val_ann_path))
    val_img_id_to_path = {
        int(im["id"]): LOCAL_DATA_DIR / "valid" / im["file_name"]
        for im in val_coco["images"]
    }

    val_preds = []
    for iid, img_path in tqdm(val_img_id_to_path.items(), total=len(val_img_id_to_path), desc="regular leaked val"):
        val_preds.extend(predict_one_image_regular(model, img_path, iid, REGULAR_CONF))

    val_pred_path = Path(f"/content/val_predictions_{MODEL_TAG}_{REG_TAG}.json")
    val_pred_path.write_text(json.dumps(val_preds))
    print("val detections:", len(val_preds))
    print("wrote:", val_pred_path)

    coco_gt = COCO(str(val_ann_path))
    coco_dt = coco_gt.loadRes(str(val_pred_path)) if val_preds else coco_gt.loadRes([])
    ev = COCOeval(coco_gt, coco_dt, "bbox")
    ev.params.maxDets = [1, 10, 500]
    ev.evaluate()
    ev.accumulate()
    ev.summarize()
else:
    print("ENABLE_REGULAR_VALIDATION=False -> skipping leaked sanity validation.")


regular leaked val:   0%|          | 0/932 [00:00<?, ?it/s]

val detections: 279535
wrote: /content/val_predictions_rfdetr_large_trainval_os_1120_32ep_regular_conf001.json
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.97s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.11s).
Accumulating evaluation results...
DONE (t=2.05s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=500 ] = 0.998
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=500 ] = 0.962
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=500 ] = 0.691
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=500 ] = 0.856
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=500 ] = 0.976
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.837
 Average Recall     (AR) @[ IoU=0

## 9. Regular full-frame inference on test_testphase keyframes

In [13]:
if ENABLE_REGULAR_INFERENCE:
    regular_preds = []
    for iid, img_path in tqdm(test_image_id_to_path.items(), total=len(test_image_id_to_path), desc="regular test"):
        regular_preds.extend(predict_one_image_regular(model, img_path, iid, REGULAR_CONF))

    write_submission(
        regular_preds,
        LOCAL_REG_PRED_PATH,
        LOCAL_REG_ZIP_OUT,
        DRIVE_REG_PRED_PATH,
        DRIVE_REG_ZIP_OUT,
    )
    validate_submission_file(LOCAL_REG_PRED_PATH, LOCAL_REG_ZIP_OUT)
    print("\nUpload regular RF-DETR zip:")
    print(DRIVE_REG_ZIP_OUT)
else:
    print("ENABLE_REGULAR_INFERENCE=False -> skipping regular inference.")


regular test:   0%|          | 0/4763 [00:00<?, ?it/s]

submission validation passed
detections: 1428448
predicted images: 4763 / 4763
class counts: {'bee': 772329, 'bumblebee': 187560, 'hoverfly': 357821, 'moth': 110738}
zip contents: ['predictions.json']

Upload regular RF-DETR zip:
/content/drive/MyDrive/BuzzSpot/submissions/submission_rfdetr_large_trainval_os_1120_32ep_regular_conf001.zip


In [14]:
# Analyze confidence thresholds on leaked validation predictions,
# then optionally create a filtered top-100 test submission.
#
# First run:
#     SELECTED_THRESHOLD = None
#
# After reviewing the table:
#     SELECTED_THRESHOLD = 0.05  # example only; do not choose blindly

import json
import zipfile
import shutil
from collections import defaultdict, Counter
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


THRESHOLDS = [
    0.001,
    0.005,
    0.01,
    0.02,
    0.03,
    0.05,
    0.075,
    0.10,
    0.15,
    0.20,
    0.30,
]

MAX_DETS_PER_IMAGE = 100
SELECTED_THRESHOLD = 0.01

def load_json_from_first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            print("Loading:", path)
            return json.loads(path.read_text())

    raise FileNotFoundError(
        "Could not find any of these files:\n"
        + "\n".join(str(path) for path in paths)
    )


def filter_and_cap_predictions(
    predictions,
    confidence_threshold,
    max_detections_per_image=100,
):
    grouped = defaultdict(list)

    for prediction in predictions:
        if float(prediction["score"]) >= confidence_threshold:
            grouped[int(prediction["image_id"])].append(prediction)

    filtered = []

    for image_id, image_predictions in grouped.items():
        image_predictions = sorted(
            image_predictions,
            key=lambda prediction: float(prediction["score"]),
            reverse=True,
        )

        filtered.extend(
            image_predictions[:max_detections_per_image]
        )

    return filtered


def prediction_count_stats(predictions, all_image_ids):
    counts = Counter(
        int(prediction["image_id"])
        for prediction in predictions
    )

    values = np.array(
        [counts.get(int(image_id), 0) for image_id in all_image_ids],
        dtype=np.int32,
    )

    return {
        "detections": int(len(predictions)),
        "mean_per_image": float(values.mean()),
        "median_per_image": float(np.median(values)),
        "max_per_image": int(values.max()) if len(values) else 0,
        "zero_images": int((values == 0).sum()),
    }


def evaluate_coco_predictions(coco_gt, predictions):
    # Suppress COCO's repeated console output during the sweep.
    with redirect_stdout(StringIO()):
        coco_dt = coco_gt.loadRes(predictions if predictions else [])

        evaluator = COCOeval(coco_gt, coco_dt, "bbox")
        evaluator.params.maxDets = [1, 10, 100]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    precision = evaluator.eval["precision"]

    per_class_ap = {}

    for class_index, class_name in enumerate(CLASS_NAMES):
        class_precision = precision[
            :, :, class_index, 0, -1
        ]

        valid_precision = class_precision[class_precision > -1]

        per_class_ap[class_name] = (
            float(valid_precision.mean())
            if valid_precision.size
            else float("nan")
        )

    return {
        "mAP50_95": float(evaluator.stats[0]),
        "mAP50": float(evaluator.stats[1]),
        "mAP75": float(evaluator.stats[2]),
        "mAR100": float(evaluator.stats[8]),
        **{
            f"AP_{class_name}": ap
            for class_name, ap in per_class_ap.items()
        },
    }


# Load raw leaked-validation predictions generated in Section 8.
raw_val_prediction_path = Path(
    f"/content/val_predictions_{MODEL_TAG}_{REG_TAG}.json"
)

raw_val_predictions = load_json_from_first_existing(
    raw_val_prediction_path,
)

# Load raw test predictions generated in Section 9.
raw_test_predictions = load_json_from_first_existing(
    LOCAL_REG_PRED_PATH,
    DRIVE_REG_PRED_PATH,
)

val_annotation_path = (
    LOCAL_DATA_DIR / "valid" / "_annotations.coco.json"
)

with redirect_stdout(StringIO()):
    val_coco_gt = COCO(str(val_annotation_path))

val_image_ids = sorted(val_coco_gt.getImgIds())

sweep_rows = []

for threshold in THRESHOLDS:
    filtered_val_predictions = filter_and_cap_predictions(
        raw_val_predictions,
        confidence_threshold=threshold,
        max_detections_per_image=MAX_DETS_PER_IMAGE,
    )

    metrics = evaluate_coco_predictions(
        val_coco_gt,
        filtered_val_predictions,
    )

    count_stats = prediction_count_stats(
        filtered_val_predictions,
        val_image_ids,
    )

    sweep_rows.append({
        "threshold": threshold,
        **metrics,
        **count_stats,
    })


sweep_df = pd.DataFrame(sweep_rows)

display_columns = [
    "threshold",
    "mAP50_95",
    "mAP50",
    "mAP75",
    "mAR100",
    "AP_bee",
    "AP_bumblebee",
    "AP_hoverfly",
    "AP_moth",
    "detections",
    "mean_per_image",
    "median_per_image",
    "max_per_image",
    "zero_images",
]

sweep_df = sweep_df[display_columns]

display(
    sweep_df.style.format({
        "threshold": "{:.3f}",
        "mAP50_95": "{:.4f}",
        "mAP50": "{:.4f}",
        "mAP75": "{:.4f}",
        "mAR100": "{:.4f}",
        "AP_bee": "{:.4f}",
        "AP_bumblebee": "{:.4f}",
        "AP_hoverfly": "{:.4f}",
        "AP_moth": "{:.4f}",
        "mean_per_image": "{:.2f}",
        "median_per_image": "{:.1f}",
    })
)

summary_path = (
    SUBMISSIONS_DIR
    / f"threshold_sweep_{MODEL_TAG}_top100.csv"
)

sweep_df.to_csv(summary_path, index=False)
print("Saved threshold summary:", summary_path)


if SELECTED_THRESHOLD is None:
    print(
        "\nNo submission created yet.\n"
        "Review the table, set SELECTED_THRESHOLD to one of the "
        "tested values, and rerun this cell."
    )

else:
    if SELECTED_THRESHOLD not in THRESHOLDS:
        raise ValueError(
            "SELECTED_THRESHOLD must be one of the tested THRESHOLDS."
        )

    filtered_test_predictions = filter_and_cap_predictions(
        raw_test_predictions,
        confidence_threshold=SELECTED_THRESHOLD,
        max_detections_per_image=MAX_DETS_PER_IMAGE,
    )

    test_counts = Counter(
        int(prediction["image_id"])
        for prediction in filtered_test_predictions
    )

    assert not test_counts or max(test_counts.values()) <= MAX_DETS_PER_IMAGE

    threshold_tag = (
        f"conf{int(round(SELECTED_THRESHOLD * 1000)):03d}"
        f"_top{MAX_DETS_PER_IMAGE}"
    )

    local_filtered_prediction_path = Path(
        f"/content/predictions_{MODEL_TAG}_regular_{threshold_tag}.json"
    )

    local_filtered_zip_path = Path(
        f"/content/submission_{MODEL_TAG}_regular_{threshold_tag}.zip"
    )

    drive_filtered_prediction_path = (
        SUBMISSIONS_DIR / local_filtered_prediction_path.name
    )

    drive_filtered_zip_path = (
        SUBMISSIONS_DIR / local_filtered_zip_path.name
    )

    write_submission(
        filtered_test_predictions,
        local_filtered_prediction_path,
        local_filtered_zip_path,
        drive_filtered_prediction_path,
        drive_filtered_zip_path,
    )

    validate_submission_file(
        local_filtered_prediction_path,
        local_filtered_zip_path,
    )

    test_stats = prediction_count_stats(
        filtered_test_predictions,
        keyframe_ids,
    )

    print("\nFiltered test statistics:")
    print(test_stats)

    print("\nCorrected submission zip:")
    print(drive_filtered_zip_path)

Loading: /content/val_predictions_rfdetr_large_trainval_os_1120_32ep_regular_conf001.json
Loading: /content/predictions_rfdetr_large_trainval_os_1120_32ep_regular_conf001.json


,threshold,mAP50_95,mAP50,mAP75,mAR100,AP_bee,AP_bumblebee,AP_hoverfly,AP_moth,detections,mean_per_image,median_per_image,max_per_image,zero_images
0,0.001,0.8740,0.9978,0.9621,0.8934,0.8498,0.9487,0.7569,0.9407,93200,100.00,100.0,100,0
1,0.005,0.8740,0.9978,0.9621,0.8931,0.8498,0.9487,0.7569,0.9407,59028,63.33,61.0,100,0
2,0.010,0.8739,0.9975,0.9621,0.8904,0.8494,0.9487,0.7567,0.9407,7649,8.21,7.0,58,0
3,0.020,0.8738,0.9975,0.9614,0.8899,0.8489,0.9487,0.7567,0.9407,1776,1.91,1.0,20,0
4,0.030,0.8738,0.9975,0.9614,0.8899,0.8489,0.9487,0.7567,0.9407,1310,1.41,1.0,13,0
5,0.050,0.8738,0.9975,0.9614,0.8899,0.8489,0.9487,0.7567,0.9407,1173,1.26,1.0,8,0
6,0.075,0.8738,0.9975,0.9614,0.8899,0.8489,0.9487,0.7567,0.9407,1144,1.23,1.0,6,0
7,0.100,0.8738,0.9975,0.9614,0.8899,0.8489,0.9487,0.7567,0.9407,1134,1.22,1.0,6,0
8,0.150,0.8738,0.9975,0.9614,0.8899,0.8489,0.9487,0.7567,0.9407,1124,1.21,1.0,6,0
9,0.200,0.8738,0.9975,0.9614,0.8899,0.8489,0.9487,0.7567,0.9407,1122,1.20,1.0,6,0


Saved threshold summary: /content/drive/MyDrive/BuzzSpot/submissions/threshold_sweep_rfdetr_large_trainval_os_1120_32ep_top100.csv
submission validation passed
detections: 118052
predicted images: 4758 / 4763
class counts: {'bee': 67565, 'bumblebee': 10739, 'hoverfly': 25199, 'moth': 14549}
zip contents: ['predictions.json']

Filtered test statistics:
{'detections': 118052, 'mean_per_image': 24.785219399538107, 'median_per_image': 19.0, 'max_per_image': 100, 'zero_images': 5}

Corrected submission zip:
/content/drive/MyDrive/BuzzSpot/submissions/submission_rfdetr_large_trainval_os_1120_32ep_regular_conf010_top100.zip


## 10. Optional SAHI helpers

In [15]:
def tile_starts(length, tile_size, overlap):
    if length <= tile_size:
        return [0]
    step = max(1, int(tile_size * (1 - overlap)))
    starts = list(range(0, length - tile_size + 1, step))
    last = length - tile_size
    if starts[-1] != last:
        starts.append(last)
    return starts

def bbox_xywh_to_xyxy(b):
    x, y, w, h = b
    return [x, y, x + w, y + h]

def ios_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = min(area_a, area_b)
    return inter / denom if denom > 0 else 0.0

def greedy_nmm_ios(records, match_threshold=0.5, max_det=None):
    merged_all = []
    for cat in [1, 2, 3, 4]:
        items = [r for r in records if int(r["category_id"]) == cat]
        items = sorted(items, key=lambda r: float(r["score"]), reverse=True)

        while items:
            head = items.pop(0)
            head_box = bbox_xywh_to_xyxy(head["bbox"])
            group = [head]
            keep = []

            for r in items:
                if ios_xyxy(head_box, bbox_xywh_to_xyxy(r["bbox"])) >= match_threshold:
                    group.append(r)
                else:
                    keep.append(r)
            items = keep

            scores = np.array([max(float(g["score"]), 1e-6) for g in group], dtype=float)
            boxes = np.array([bbox_xywh_to_xyxy(g["bbox"]) for g in group], dtype=float)
            weights = scores / scores.sum()
            merged_box = (boxes * weights[:, None]).sum(axis=0).tolist()
            score = max(float(g["score"]) for g in group)

            clipped = clip_xyxy(*merged_box, W_FULL, H_FULL)
            if clipped is None:
                continue
            x1, y1, x2, y2 = clipped
            merged_all.append({
                "image_id": int(group[0]["image_id"]),
                "category_id": cat,
                "bbox": [round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)],
                "score": round(score, 5),
            })

    merged_all = sorted(merged_all, key=lambda r: float(r["score"]), reverse=True)
    if max_det is not None:
        merged_all = merged_all[:max_det]
    return merged_all

def predict_one_image_sahi(model, img_path, image_id, threshold=0.05):
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    assert W == W_FULL and H == H_FULL, f"Unexpected image size {W}x{H}: {img_path}"

    xs = tile_starts(W, SAHI_SLICE_SIZE, SAHI_OVERLAP)
    ys = tile_starts(H, SAHI_SLICE_SIZE, SAHI_OVERLAP)
    all_records = []
    tile_path = Path("/content/_rfdetr_sahi_tile.jpg")

    for y0 in ys:
        for x0 in xs:
            x1 = min(x0 + SAHI_SLICE_SIZE, W)
            y1 = min(y0 + SAHI_SLICE_SIZE, H)
            tile = img.crop((x0, y0, x1, y1))
            tile.save(tile_path, quality=95)

            det = model.predict(str(tile_path), threshold=threshold)
            all_records.extend(detections_to_records(det, image_id=image_id, x_offset=x0, y_offset=y0, W=W, H=H, max_det=300))

    return greedy_nmm_ios(
        all_records,
        match_threshold=SAHI_MATCH_THRESHOLD,
        max_det=SAHI_MAX_DET_PER_IMAGE,
    )


## 11. Optional SAHI inference on test_testphase keyframes

In [16]:
if ENABLE_SAHI_INFERENCE:
    sahi_preds = []
    for iid, img_path in tqdm(test_image_id_to_path.items(), total=len(test_image_id_to_path), desc="SAHI test"):
        sahi_preds.extend(predict_one_image_sahi(model, img_path, iid, SAHI_CONF))

    write_submission(
        sahi_preds,
        LOCAL_SAHI_PRED_PATH,
        LOCAL_SAHI_ZIP_OUT,
        DRIVE_SAHI_PRED_PATH,
        DRIVE_SAHI_ZIP_OUT,
    )
    validate_submission_file(LOCAL_SAHI_PRED_PATH, LOCAL_SAHI_ZIP_OUT)
    print("\nUpload SAHI RF-DETR zip:")
    print(DRIVE_SAHI_ZIP_OUT)
else:
    print("ENABLE_SAHI_INFERENCE=False -> skipping SAHI inference.")


ENABLE_SAHI_INFERENCE=False -> skipping SAHI inference.


## 12. Output summary

In [17]:
print("Selected RF-DETR checkpoint:", DRIVE_SELECTED_WEIGHTS)
print("Regular submission:", DRIVE_REG_ZIP_OUT if ENABLE_REGULAR_INFERENCE else "regular inference disabled")
print("SAHI submission:", DRIVE_SAHI_ZIP_OUT if ENABLE_SAHI_INFERENCE else "SAHI inference disabled")
print()
print("If OOM: set BATCH_SIZE=1 and GRAD_ACCUM_STEPS=16, or reduce RESOLUTION to 896.")
print("Custom classes are mapped by detections.data['class_name']; numeric RF-DETR IDs are not guessed.")


Selected RF-DETR checkpoint: /content/drive/MyDrive/BuzzSpot/weights/rfdetr_large_trainval_os_1120_32ep_selected_for_inference.pth
Regular submission: /content/drive/MyDrive/BuzzSpot/submissions/submission_rfdetr_large_trainval_os_1120_32ep_regular_conf001.zip
SAHI submission: SAHI inference disabled

If OOM: set BATCH_SIZE=1 and GRAD_ACCUM_STEPS=16, or reduce RESOLUTION to 896.
Custom classes are mapped by detections.data['class_name']; numeric RF-DETR IDs are not guessed.


In [20]:
# Show full-frame leaked-val images where RF-DETR missed at least one hoverfly.
# Left: GT
# Middle: predictions
# Right: readable text summary / legend

import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image


# --------------------------------------------------
# Config
# --------------------------------------------------

VIS_CONF = 0.01
VIS_MAX_DETS = 100
IOU_MATCH_THRESH = 0.50

MAX_IMAGES_TO_SHOW = 12          # set None to show all
SORT_BY_NUM_MISSED = True        # most missed hoverflies first

CATEGORY_NAME = {
    1: "bee",
    2: "bumblebee",
    3: "hoverfly",
    4: "moth",
}

CATEGORY_COLOR = {
    1: "gold",
    2: "red",
    3: "limegreen",
    4: "dodgerblue",
}

MISSED_HOVERFLY_COLOR = "magenta"


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def xywh_to_xyxy(bbox):
    x, y, w, h = map(float, bbox)
    return [x, y, x + w, y + h]

def bbox_area_xywh(bbox):
    _, _, w, h = map(float, bbox)
    return max(0.0, w) * max(0.0, h)

def iou_xywh(box_a, box_b):
    ax1, ay1, ax2, ay2 = xywh_to_xyxy(box_a)
    bx1, by1, bx2, by2 = xywh_to_xyxy(box_b)

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter = inter_w * inter_h

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter

    return 0.0 if union <= 0 else inter / union

def greedy_match_hoverflies(gt_hoverflies, pred_hoverflies, iou_thresh=0.5):
    """
    gt_hoverflies: list of annotations (GT hoverfly only)
    pred_hoverflies: list of predictions (pred hoverfly only)
    returns:
      matched_gt_indices, matched_pred_indices, missed_gt_indices
    """
    candidate_pairs = []

    for gt_idx, gt in enumerate(gt_hoverflies):
        for pred_idx, pred in enumerate(pred_hoverflies):
            iou = iou_xywh(gt["bbox"], pred["bbox"])
            if iou >= iou_thresh:
                candidate_pairs.append((iou, gt_idx, pred_idx))

    # highest IoU first
    candidate_pairs.sort(reverse=True, key=lambda x: x[0])

    matched_gt = set()
    matched_pred = set()

    for iou, gt_idx, pred_idx in candidate_pairs:
        if gt_idx in matched_gt or pred_idx in matched_pred:
            continue
        matched_gt.add(gt_idx)
        matched_pred.add(pred_idx)

    missed_gt = [i for i in range(len(gt_hoverflies)) if i not in matched_gt]

    return matched_gt, matched_pred, missed_gt

def filter_predictions(predictions, conf_thresh=0.01, max_dets=100):
    by_image = defaultdict(list)

    for pred in predictions:
        if float(pred["score"]) >= conf_thresh:
            by_image[int(pred["image_id"])].append(pred)

    filtered = {}
    for image_id, preds in by_image.items():
        filtered[image_id] = sorted(
            preds,
            key=lambda p: float(p["score"]),
            reverse=True,
        )[:max_dets]

    return filtered

def add_box(ax, bbox, color, linewidth=2.5, linestyle="-", label_text=None, text_color=None):
    x, y, w, h = map(float, bbox)
    rect = Rectangle(
        (x, y), w, h,
        fill=False,
        edgecolor=color,
        linewidth=linewidth,
        linestyle=linestyle,
    )
    ax.add_patch(rect)

    if label_text:
        ax.text(
            x,
            max(5, y - 6),
            label_text,
            fontsize=11,
            color=text_color or color,
            bbox=dict(facecolor="black", alpha=0.7, pad=1.5, edgecolor="none"),
        )


# --------------------------------------------------
# Load val annotations
# --------------------------------------------------

val_annotation_path = LOCAL_DATA_DIR / "valid" / "_annotations.coco.json"
with open(val_annotation_path, "r") as f:
    val_coco = json.load(f)

val_images = {int(img["id"]): img for img in val_coco["images"]}

gt_by_image = defaultdict(list)
for ann in val_coco["annotations"]:
    gt_by_image[int(ann["image_id"])].append(ann)


# --------------------------------------------------
# Load saved validation predictions
# --------------------------------------------------

raw_val_prediction_path = Path(f"/content/val_predictions_{MODEL_TAG}_{REG_TAG}.json")
assert raw_val_prediction_path.exists(), f"Missing val predictions file: {raw_val_prediction_path}"

with open(raw_val_prediction_path, "r") as f:
    raw_val_predictions = json.load(f)

pred_by_image = filter_predictions(
    raw_val_predictions,
    conf_thresh=VIS_CONF,
    max_dets=VIS_MAX_DETS,
)


# --------------------------------------------------
# Find images with at least one missed hoverfly
# --------------------------------------------------

missed_cases = []

for image_id, anns in gt_by_image.items():
    gt_hoverflies = [ann for ann in anns if int(ann["category_id"]) == 3]
    if not gt_hoverflies:
        continue

    preds = pred_by_image.get(image_id, [])
    pred_hoverflies = [pred for pred in preds if int(pred["category_id"]) == 3]

    matched_gt, matched_pred, missed_gt = greedy_match_hoverflies(
        gt_hoverflies,
        pred_hoverflies,
        iou_thresh=IOU_MATCH_THRESH,
    )

    if len(missed_gt) > 0:
        missed_cases.append({
            "image_id": image_id,
            "gt_hoverflies": gt_hoverflies,
            "pred_hoverflies": pred_hoverflies,
            "missed_gt_indices": missed_gt,
            "matched_gt_indices": matched_gt,
            "matched_pred_indices": matched_pred,
            "all_gt": anns,
            "all_preds": preds,
        })

if SORT_BY_NUM_MISSED:
    missed_cases.sort(
        key=lambda x: (
            len(x["missed_gt_indices"]),
            len(x["gt_hoverflies"]),
        ),
        reverse=True,
    )

if MAX_IMAGES_TO_SHOW is not None:
    missed_cases = missed_cases[:MAX_IMAGES_TO_SHOW]

print(f"Images with at least one missed hoverfly: {len(missed_cases)} shown")


# --------------------------------------------------
# Visualize
# --------------------------------------------------

for case_idx, case in enumerate(missed_cases, start=1):
    image_id = case["image_id"]
    image_info = val_images[image_id]
    image_path = LOCAL_DATA_DIR / "valid" / image_info["file_name"]

    image = Image.open(image_path).convert("RGB")
    gt_anns = case["all_gt"]
    pred_anns = case["all_preds"]

    fig = plt.figure(figsize=(24, 9))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 1.15, 0.9])

    ax_gt = fig.add_subplot(gs[0, 0])
    ax_pred = fig.add_subplot(gs[0, 1])
    ax_txt = fig.add_subplot(gs[0, 2])

    # --- GT panel
    ax_gt.imshow(image)
    ax_gt.set_title(
        f"Ground truth — image {image_id}\n"
        f"GT hoverflies: {len(case['gt_hoverflies'])}, missed: {len(case['missed_gt_indices'])}",
        fontsize=15
    )
    ax_gt.axis("off")

    # Draw all GT boxes lightly
    for ann in gt_anns:
        cid = int(ann["category_id"])
        color = CATEGORY_COLOR[cid]
        add_box(
            ax_gt,
            ann["bbox"],
            color=color,
            linewidth=2.0 if cid != 3 else 2.5,
            label_text=CATEGORY_NAME[cid],
            text_color="white",
        )

    # Highlight missed hoverflies strongly + number them
    missed_hoverflies = [case["gt_hoverflies"][i] for i in case["missed_gt_indices"]]
    for missed_idx, ann in enumerate(missed_hoverflies, start=1):
        x, y, w, h = ann["bbox"]
        add_box(
            ax_gt,
            ann["bbox"],
            color=MISSED_HOVERFLY_COLOR,
            linewidth=4.0,
            linestyle="-",
            label_text=f"MISSED H{missed_idx}",
            text_color="white",
        )

    # --- Prediction panel
    ax_pred.imshow(image)
    ax_pred.set_title(
        f"RF-DETR predictions — conf ≥ {VIS_CONF}\n"
        f"predictions shown: {len(pred_anns)}",
        fontsize=15
    )
    ax_pred.axis("off")

    for pred in pred_anns:
        cid = int(pred["category_id"])
        color = CATEGORY_COLOR[cid]
        score = float(pred["score"])
        add_box(
            ax_pred,
            pred["bbox"],
            color=color,
            linewidth=2.2,
            label_text=f"{CATEGORY_NAME[cid]} {score:.2f}",
            text_color="white",
        )

    # Also outline missed GT hoverflies on prediction panel so you can inspect those locations
    for missed_idx, ann in enumerate(missed_hoverflies, start=1):
        add_box(
            ax_pred,
            ann["bbox"],
            color=MISSED_HOVERFLY_COLOR,
            linewidth=4.0,
            linestyle="--",
            label_text=f"GT H{missed_idx}",
            text_color="white",
        )

    # --- Text panel
    ax_txt.axis("off")

    lines = []
    lines.append(f"Image {image_id}")
    lines.append("")
    lines.append(f"Missed hoverflies: {len(missed_hoverflies)}")
    lines.append(f"Total GT hoverflies: {len(case['gt_hoverflies'])}")
    lines.append(f"Pred hoverflies: {len(case['pred_hoverflies'])}")
    lines.append("")

    lines.append("Missed hoverfly GT boxes:")
    for missed_idx, ann in enumerate(missed_hoverflies, start=1):
        x, y, w, h = map(float, ann["bbox"])
        lines.append(
            f"H{missed_idx}: x={x:.1f}, y={y:.1f}, w={w:.1f}, h={h:.1f}"
        )

    lines.append("")
    lines.append("Predicted hoverflies:")
    if len(case["pred_hoverflies"]) == 0:
        lines.append("none")
    else:
        for pred_idx, pred in enumerate(case["pred_hoverflies"], start=1):
            x, y, w, h = map(float, pred["bbox"])
            score = float(pred["score"])
            lines.append(
                f"P{pred_idx}: score={score:.3f}, "
                f"x={x:.1f}, y={y:.1f}, w={w:.1f}, h={h:.1f}"
            )

    ax_txt.text(
        0.0, 1.0,
        "\n".join(lines),
        va="top",
        ha="left",
        fontsize=13,
        family="monospace",
    )

    plt.tight_layout()
    plt.show()

Images with at least one missed hoverfly: 0 shown
